   ... binary bomb lab is genuinely one of the most fun coursework exercises in
   computing -- it's like a puzzle boc where each phase teaches you something
   new about how programs actually work at the machine level.

   ...

   A quick overview of the approach we'll take for each phase: we'll disassemble
   the bomb with `objdump -d`, read the assembly to understand the logic, and
   use `gbb` to verify

   -- ... classic "initiation" hurdles... Don't worry... bomb isn't broken -- it;s
   just a matter of Linux permissions and how GDB communicates with the program. 

...

1. FIX "PERMISSION DENIED"
   The error `Permission denied`... means the file currently doesn't have 
   "execute" permissions. Even though you downloaded it, Linux needs to be 
   explicitly told that this file is a program it it allowed to run.

   THE FIX:
      Run this command in your terminal...

```Bash
chmod +x bomb
```
   `chmod +x` stands for "change mode + execute." This will allow the file to 
   run.


2. FIX "Undefined Command: `testing123`"
   You are trying to type your input directly into the `(gdb)` prompt. GBP
   thinks `testing123` is a command like `run` or `break`.

   THE CONCEPT:
      Inside GDB, there is a clear distinction between DEBUGGER COMMANDS (to
      control GDB) and PROGRAM INPUT (data you send to the bomb).

   THE CORRECT WORKFLOW:
      1. Start GDB: `gdb ./bomb`
      2. Set your safety net: `break explode_bomb`
      3. Set the phase start: `break phase_1`
      4. Run the program: Type `run` and hit Enter.
      5. Wait for the prompt: The bomb will print "Welcome to my fiendish little
         bomb..." and then pause. ONLY THEN do you type `testing123` and hit 
         Enter.      

---

   `objdump -d` is a command-line utility used to DISASSEMBLE a binary file. In
   the context..., it is the primary tool for turning the "unreadable" machine
   code (the `bomb` executable) back into human-readable assembly language.


WHAT IT DOES
   When Dr. Evil compiled the bomb, his C code was translated into a sequence of 
   bytes (machine code) that the CPU executes. Because you weren't given the 
   source code for the phases, `objdump -d` allows you to "reverse engineer" 
   the logic by showing you the assembly instructions those bytes represent.   


BREAKDOWN OF THE OUTPUT
   If you look at the screenshot you provided, or if you run the command 
   yourself, the output is typically organised into three columns:
      1. ADDRESS: The memory location where the instruction lives (e.g., `3f17`
         in your image).
      2. HEX DUMP (Opcodes): The actual bytes of the machine code (e.g., `55`,
         `48 89 e5`).
      3. INSTRUCTION: The human-readable assembly mnemonic and its operands 
         (e.g., `push %rbp` or `mov %rsp, %rbp`).


WHY IT IS USED IN THE BOMB LAB
   * UNDERSTANDING BEHAVIOR: By reading the assembly, you can see how the 
     program treats your input. For example, you can find where it compares your
     string to a "secret" string stored in memory.
   * LOCATING FUNCTIONS: You can use it to find the entry points for specific
     phases, such as `<phase_1>` or `<phase_2>`.
   * IDENTIFY SYSTEM CALLS: While `objdump -d` can sometimes by cryptic with
     system-level functions like `sscanf`, it provides the "skeleton" of the
     program's logic.     


HOW TO USE IT EFFECTIVELY
   To make the output easier to manage, student often redirect the results to a
   text file so they can search through it using a text editor ...

   `objdump -d bomb > assembly.txt`

      NOTE: The coursework specification mentions that while `objdump -d` is
      powerful, it doesn't tell the "whole story" for certain function calls. 
      For those, you'll need to use `gdb` to "disassemble within the debugger" 
      to see more detail. 

-- GNU/Linux is a FOSS, UNIX-like operating system composed of GNU software 
   tools (compilers, libraries, shell) and the Linux kernel, which manages 
   hardware resources. It enables users to run applications, manage files, and
   develop software, commonly distrbuted as packages like Ubuntu or Debian. It
   is often used for web servers, development, and desktop computing.

   
   KEY COMPONENTS AND USAGE
      * GNU Components: Provides essential userland tools, including the GCC
        compiler, Bash shell, and core utilities.
      * Linux Kernel: Manages system hardware, memory and processor resources,
      * Usage Examples: It powers nearly all top internet servers, powers 
        Android devices, runs desktop environments (GNOME, KDE), and is used in
        embedded systems.


  SYNONYMS AND NAMING
     * Commonly referred to as: Linux, Linux distribution, open-source OS.
     ...      


---

   ... hands dirty in the debugger is actually the most rewarding way to learn
   how the machine really thinks.

   Using `gdb` (the GNU Debugger) is highly recommended for this coursework
   because it lets you freeze time, inspect the CPU's exact state, and safely
   step through the bomb's logic without accidentally detonating it. Remember, 
   the specification warns that every time the bomb explodes, it sends a message
   to the lab server and you lose half a point. 

   Let's walk through the essential `gdb` workflow, step-by-step, to get you
   safely into Phase 1.


---
STEP 1: Launch GDB and Set Your Safety Nets   
   First, start the debugger with your bomb executable.

```Bash
gdb ./bomb
```

   Before you do anything else, you need to set breakpoints. A breakpoint tells
   `gdb` to pause execution right before it runs a specific function.

   Inside the `(gdb)` prompt, type:

```
(gdb) break explode_bomb
```

   This is your ultimate safety net. If you enter the wrong string, the bomb
   will try to call `explode_bomb`, but `gdb` will freeze it right before it
   executes, saving your grade!

   Next, set a breakpoint for the start of the first phase:

```
(gdb) break phase_1
```
   

---
STEP 2: Get the "Hacker" UI (TUI Mode)   
   Since you liked the layout your friend was using, let's turn that on. This is
   called the Text User Interface (TUI).

```
(gdb) layout asm
(gdb) layout regs
```

   This splits your terminal screen. The top window will show the CPU registers
   (like `%rax`, `%rdi`, etc.), the middle window will show the assembly code 
   highlighting exactly where you are, and the bottom window is your command 
   prompt.


---
STEP 3: Run the Program
   Now that your safety nets are up, start the program:

```
(gdb) run
```
   The program will start, print Dr Evil's welcome message, and then pause, 
   waiting for your input. Type in a dummy test string (e.g., `Testing123`) and
   hit enter.

   Because we set a breakpoint earlier, `gdb` will immediately pause the program 
   the moment it enters `<phase_1>` and give control back to you. 


---
STEP 4: Inspecting and Stepping (The Core Loop)
   You are now frozen inside Phase 1. Here are the three most important
   commands you will use to navigate and investigate:


1. DISASSEMBLE THE CURRENT FUNCTION:
   If you ever lose your place or aren't using the TUI mode, you can view the
   assembly for the function you are currently in.    
```
(gdb) disas
```

2. STEPPING FORWARD:
   To execute the assembly code one single line at a time, use "next 
   instruction":
```
(gdb) ni
```
   (Note: You can just press the `Enter` key on your keybord to repeat the
   last command, so you don't have to keep typing `ni`!)
   

3. INSPECTING MEMORY (Looking for the secret string)
   As you step through, you will likely see an instruction where an address is
   moved into a register right before a function call like `string_not_equal`.
   It might look like `mov $0x402400, %esi`.

   You can inspect what is actually stored at that memory address. To examine a
   string at a specific hex address, use:

```
(gdb) x/s 0x402400
```
   
   (Replace `0x402400` with whatever memory address you actually see in your 
   code). If you see a readable phrase pop up, congratulations, you've likely
   just found the password for that phrase!












---

In [ ]:
(\o/)___________________________________________________________(\o/)
(/|\)                                                           (/|\)
  |                                          .-~~~-.              |
  |                                        /        }             |
  |                                       /      .-~              |
  |                             \        |        }               |
  |             __   __       ___\.~~-.-~|     . -~_              |
  |            / \./  \/\_       { O |  ` .-~.    ;  ~-.__        |
  |        __{^\_ _}_   )  }/^\   ~--~/-|_\|   :   : .-~          |
  |       /  /\_/^\._}_/  //  /     /   |  \~ - - ~               |
  |      (  (__{(@)}\__}.//_/__A__/_A___|__A_\___A______A_____A   |
  |       \__/{/(_)\_}  )\\ \\---v-----V----v----v-----V-----v--- |
  |         (   (__)_)_/  )\ \>                                   |
  |          \__/     \__/\/\/                                    |
  |             \__,--'                                           |
  |                                                               |
(\o/)___________________________________________________________(\o/)
(/|\)                                                           (/|\)


---

-- `grep` is one of the most essential tools in a programmer's toolkit, 
   especially when you're navigating a large project like the Binary Bomb, it
   stands for Global Regular Expression Print.

   Essentially, it is a "search" command for your terminal.


   HOW `grep` WORKS
      At its simplest, `grep` looks through a file (or multiple files) and 
      prints every line that contains a specific pattern or piece of text.

   THE SYNTAX:
      `grep "pattern" filename`

   EXAMPLE:
      If you want to find every time Dr. Evil is mentioned in your bomb's C 
      code:
      `grep "Dr. Evil" bomb.c`


   ---
   HOW `grep -A` WORKS
      The `-A` flag stands for "AFTER." It is used when you don't just want the 
      line that matches your search, but you also want to see the context that
      follows it.

   THE SYNTAX:
      `grep -A <number> "pattern" filename`
         * `-A` 5: This tells `grep` to show you the matching line pLUS THE 5 
           LINES IMMEDIATELY FOLLOWING IT.
         * WHY USE IT? In assembly or C code, a single line (like a function 
           name) doesn't tell you much. You usually need to see the logic that
           follows that line to understand what the program is doing. 


   ---
   WHY AND HOW IT'S NORMALLY USED
      Since you are catching up on the module and working through this 
      coursework, `grep` will save you hours of scrolling.
         * FINDING FUNCTION DEFINITIONS: If you want to see the code for 
           `phase_1` in your disassembled assembly file:
              `grep -A 20 "<phase_1>:" assembly.txt`
           This will show you the start of the function and the next 20 
           instructions.
         * SEARCHING FOR SPECIFIC INSTRUCTIONS: If you want to find every place
           the bomb might explode, you can search for the "BOOM!!!" string or
           the function call:
              `grep -A 3 "explode_bomb" assembly.txt > grep-explode_bomb`
         * FILTERING OUTPUT: You can "pipe" other commands into `grep`. For
           example, to find your specific bomb ID in a list:
              `ls | grep "bomb"`

   SUMMARY TABLE

   Command // Purpose
   `grep "text"` // Find and print lines containing "text".
   `grep -i "text"` // IGNORE CASE (finds "Text", TEXT, and "text").
   `grep -r "text" .` // RECURSIVE search through every file in your current 
                         folder
   `grep -A 5 "text"` // Show the match and 5 LINES AFTER it.
   `grep -B 5 "text"` // Show the match and 5 LINES BEFORE it.

   In the context of ... understanding the difference between `stepi` (step
   instruction) and `nexti` (next instruction) is the difference between 
   performing a "deep dive" into every function and staying on the main path.
   While both commands execute exactly one assembly instruction at a time, they
   behave differently when they encounter a `call` instruction. `stepi` will
   "step into" the function being called, moving the debugger's (GDB) focus to
   the first instruction of that new function. This is useful if you need to see
   exactly how a helper function like `strings_not_equal` or `func4` works
   internally.

   `nexti`, on the other hand, treats a function call as a single unit of
   execution. If you use `nexti` on a `call` instruction, GDB will execute the
   entire called function at full speed and only pause again at the very next
   instruction in your current function. This is generally the "safer" and 
   faster way to navigate through a phase because it prevents you from getting
   lost in complex library code (like `printf` or `sscanf`) that isn't relevant
   to finding your input string. It allows you to observe the state of the
   registers immediately after a function returns to see what the result was.

   Regarding your ... YES, `ni` IS EXACTLY THE SAME AS `nexti`, In GDB, `ni` is
   simply a shorthand alias for the full command `nexti`, just as `si` is the
   shorthand for `stepi`. You can use these abbreviatins to save time while
   debugging. Most students prefer using `ni` for the majority of their work to
   avoid stepping into system-level functions, only switching to `si` when
   they specifically want to investigate the inner logic of one of Dr Evil's
   custom "secret" functions.

   ...

---


   A quick overview of the approach we'll take for each phase: we'll disassemble
   the bomb with `objdump -d`, read the assembly to understand the logic, and
   use `gdb` to verify our understanding without detonating anything. I'll 
   explain the x86 concepts as they come up -- things like calling conventions,
   comparison instructions, conditional jumps, stack frames, and how C 
   consutrcts like loop,s arrays, and switch statements look in assembly.

   A couple of things to get us started. 
   In the emantime, a few quick tips...

   DON'T RUN THE BOMB WITHOUT GDB. Always launch it as `gdb ./bomb` and set a
   breakpoint on `explode_bomb` first (`break explode_bomb`) -- this way if you
   get a wrong answer, you can kill it before it phones home and costs you half
   a point.

   


   let's start from the beginning... run three command and share the output
   ...


1. GET THE FULL DISASSEMBLY:
```
cd `path`
objdump -d ./bomb > assembly.txt
```

2. LOOK AT THE STRINGS EMBEDDED IN THE BOMB:
```
strings bomb
```

3. Show me the Phase 1 disassembly specifically:
```
objdump -d bomb | grep -A 30 "<phase_1>"
```

   The first command creates a full disassembly file you can refer back to 
   throughout. The second one is a quick win -- `strings` pulls out every
   readable text string baked into the binary. Phase 1 almost always involves
   comparing your input against a hidden string, so you might spot something
   interesting tthere.

   The third command shows us the assembly code for just `phase_1`, which is
   where we'll start our detective work. Once you paste the output here, I'll
   walk you through the assembly line by line -- we'll cover how function calls
   work, what registers like `%eax` and `%edi` do, how string comparisons
   happen at the machine level, and how to read the control flow to figure out 
   the answer without ever risking an explosion. 


---   


   ... dive into PHASE 1 -- the gentlest introduction to reverse engineering, 
   and it'll teach you some foundational x86-64 concepts.


### PHASE 1 -- THE STRING COMPARISON
   Here's the code we care about:

```asm
0000000000003607 <phase_1>:
    3607:	f3 0f 1e fa          	endbr64
    360b:	48 83 ec 08          	sub    $0x8,%rsp
    360f:	48 8d 35 52 2d 00 00 	lea    0x2d52(%rip),%rsi        # 6368 <_IO_stdin_used+0x1368>
    3616:	e8 4e 05 00 00       	call   3b69 <strings_not_equal>
    361b:	85 c0                	test   %eax,%eax
    361d:	75 05                	jne    3624 <phase_1+0x1d>
    361f:	48 83 c4 08          	add    $0x8,%rsp
    3623:	c3                   	ret
    3624:	e8 ca 07 00 00       	call   3df3 <explode_bomb>



0000000000003607 <phase_1>:
    3607:  endbr64
    360b:  sub    $0x8,%rsp
    360f:  lea    0x2d52(%rip),%rsi        # 6368
    3616:  call   3b69 <strings_not_equal>
    361b:  test   %eax,%eax
    361d:  jne    3624 <phase_1+0x1d>
    361f:  add    $0x8,%rsp
    3623:  ret
    3624:  call   3df3 <explode_bomb>
```

---   
   Let's walk through each line and the concepts behind them.

   `endbr64` -- This is a security feature (Intel CET / Control-flow Enforcement
   Technology). It makes valid for indirect jumps/calls. You can ignore it for
   this coursework; it's just a safety marker.

   `sub $0x8, %rsp` -- This adjusts the stack pointer to keep it 16-byte aligned
   (required by the x86-64 ABI). The `call` instruction that brought us here
   already pushed an 8-byte return address, so the `sub` brings it to a 16-byte
   boundary. 

   `lea 0x2d52(%rip)` -- This is the key line. `lea` means "Load Effective 
   Address" -- it computes an address and stores it in `%rsi`. The address is
   `%rip + 0x2d52`, which the diassembler helpfully tells us resolves to 
   `0x6368`. This is a pointer to a string stored in the binary's read-only data
   section. In the x84-64 calling convention, `%rsi` is the SECOND ARGUMENT to a
   function.

   `call 3b69 <strings_not_equal>` -- Calls a function that compares two
   strings. Remember, your input was already in `%rdi` (the FIRST ARGUMENT) --
   it was put there by `main` just before calling `phase_1`. So this is 
   essentially: `strings_not_equal(your_input, secret_string)`.

   `test %eax, %eax` -- This ANDs `%eax` with itself. The key insight: this
   doesn't change `%eax`, but it SETS THE CPU FLAgs. Specifically, if `%eax` is 
   0, the Zero Flag (ZF) is set. `%eax` holds the return value of the function
   -- 0 means the strings ARE equal, 1 means theya re not. 

   `jne 3624` -- "Jump if Not Equal" -- this checks the Zero Flag. If ZF is NOT
   set (meaning `%eax` was nonzero, meaning the strings were different), it
   jumps to `explode_bomb`. If ZF is set (strings matched), execution falls
   through to the `ret` and you survive.

   So the whole function boils down to this pseudocode:

```c   
void phase_1(char *input) {
    if (strings_not_equal(input, SECRET_STRING) != 0)
        explode_bomb();
}
```
    YOUR JOB: find what string lives at address `0x6368`.

    Since this is a PIE (Position Independent Executable), the addresses get 
    relocated at runtime, so you can't just look up `0x6368` directly. Here's
    what to do in GDB: 

```bash
gdb ./bomb
(gdb) break explode_bomb
(gdb) break phase_1
(gdb) run
```

   Type any garbage string when prompted, then when it hits the `phase_1`
   breakpoint:

```bash
(gdb) x/s $rsi
```
   
   This prints the string pointed to by `%rsi` -- which is the secret answer. 
   The `x/s` command means "examine memor as a string."


   ... The issue is that the breakpoint fires at the VERY START of `phase_1`
   before the `lea` instruction loads the secret string address into `%rsi`. At
   that point `%rsi` still holds junk from earlier -- and GDB is trying to
   intercept the ASCII bytes of your input as memory address.

   You need to step forward past the `lea` instruction first. From where you are
   right now, do this:

```bash
(gdb) stepi 2
```   

   This advances two instructions -- past the `sub` and the `lea`. Now `%rsi`
   will hold the actual pointer to the secret string. Then:

```bash
(gdb) x/s $rsi
```

   `stepi` (or `si` for short) executes exactly ONE MACHINE INSTRUCTION at a 
   time. It's one fo the most useful GDB commands for this courseworks -- it 
   lets you watch the CPU state change instruction by instruction.

   The reason this happened is a really important concept: REGISTER VALUES 
   CHANGE AS INSTRUCTIONS EXECUTE. When you set a breakpoint on a function, GDB
   stops before any of that function's instructions run. The registers still
   contain whatever the caller left in them. only after the `lea` at `0x360f`
   actually executes does `%rsi` get loaded with the secret string's address.







